# Data Cleaning Plan

EDA findings into explicit cleaning decisions before data splitting, preprocessing, and modeling.

The purpose is to document what needs to be corrected, retained, excluded from modeling, or handled later in the preprocessing pipeline.

## 1. Environment Setup

Load the core Python libraries, connect the notebook to the project `src` folder, and import the project data-loading helpers.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve().parents[0]
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from credit_risk_platform.data.german_credit import interim_german_credit_dir
from credit_risk_platform.utils.io import load_csv

print("project_root:", project_root)
print("src_dir exists:", src_dir.exists())

project_root: /Users/ememakpan/Documents/New project/applied_ai_economics_portfolio/project_01_ai_credit_risk_platform
src_dir exists: True


## 2. Load the Standardized Dataset

Load the cleaned column-name version of the German Credit dataset created during ingestion.

In [3]:
interim_dir = interim_german_credit_dir(project_root)
df = load_csv(interim_dir / "german_credit_standardized.csv")

print("Shape:", df.shape)
df.head()

Shape: (1000, 24)


,applicant_id,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_duration,installment_rate_pct_income,personal_status_sex,...,other_installment_plans,housing,existing_credits_count,job_type,liable_people_count,telephone,foreign_worker,risk_class,risk_label,TARGET
0,1,A11,6,A34,A43,1169,A65,A75,4,A93,...,A143,A152,2,A173,1,A192,A201,1,good,0
1,2,A12,48,A32,A43,5951,A61,A73,2,A92,...,A143,A152,1,A173,1,A191,A201,2,bad,1
2,3,A14,12,A34,A46,2096,A61,A74,2,A93,...,A143,A152,1,A172,2,A191,A201,1,good,0
3,4,A11,42,A32,A42,7882,A61,A74,2,A93,...,A143,A153,1,A173,2,A191,A201,1,good,0
4,5,A11,24,A33,A40,4870,A61,A73,3,A93,...,A143,A153,2,A173,2,A191,A201,2,bad,1


## 3. Cleaning Scope

Cleaning is focused on data quality and analysis readiness. It does not include model preprocessing steps such as one-hot encoding, scaling, fitted imputation, or train-test transformations.

## 4. Classify Columns by Role

Separate identifier columns, target columns, numeric features, and categorical features. This prevents leakage and keeps later modeling decisions clear.

In [4]:
identifier_cols = ["applicant_id"]
target_cols = ["risk_class", "risk_label", "TARGET"]

numeric_cols = [
    col
    for col in df.select_dtypes(include="number").columns.tolist()
    if col not in identifier_cols + target_cols
]

categorical_cols = [
    col
    for col in df.select_dtypes(exclude="number").columns.tolist()
    if col not in target_cols
]

display(pd.DataFrame({"identifier_cols": pd.Series(identifier_cols)}))
display(pd.DataFrame({"target_cols": pd.Series(target_cols)}))
display(pd.DataFrame({"numeric_cols": pd.Series(numeric_cols)}))
display(pd.DataFrame({"categorical_cols": pd.Series(categorical_cols)}))

,identifier_cols
0,applicant_id


,target_cols
0,risk_class
1,risk_label
2,TARGET


,numeric_cols
0,duration_months
1,credit_amount
2,installment_rate_pct_income
3,present_residence_years
4,age_years
5,existing_credits_count
6,liable_people_count


,categorical_cols
0,checking_account_status
1,credit_history
2,purpose
3,savings_account
4,employment_duration
5,personal_status_sex
6,other_debtors_guarantors
7,property_type
8,other_installment_plans
9,housing


## 5. Check Missing Values and Duplicate Rows

Confirm whether the standardized dataset has missing values or duplicated records that require cleaning decisions.

In [5]:
missing_table = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_pct": ((df.isna().sum() / len(df)) * 100).round(2),
    }
).sort_values("missing_count", ascending=False)

duplicate_count = int(df.duplicated().sum())

display(missing_table)
print("Duplicate rows:", duplicate_count)

,missing_count,missing_pct
applicant_id,0,0.0
checking_account_status,0,0.0
risk_label,0,0.0
risk_class,0,0.0
foreign_worker,0,0.0
telephone,0,0.0
liable_people_count,0,0.0
job_type,0,0.0
existing_credits_count,0,0.0
housing,0,0.0


Duplicate rows: 0


### Missingness and Duplicate Decision

If all missing counts are zero, no missing-value cleaning is needed at this stage. If duplicates are zero, no row deduplication is needed. Any future imputation should happen inside the preprocessing pipeline after the data split.

## 6. Review Numeric Ranges

Check whether numeric features contain impossible values such as negative duration, negative credit amount, or invalid age values.

In [6]:
numeric_summary = df[numeric_cols].describe().T
display(numeric_summary)

range_checks = pd.DataFrame(
    {
        "feature": numeric_cols,
        "min_value": [df[col].min() for col in numeric_cols],
        "max_value": [df[col].max() for col in numeric_cols],
        "unique_values": [df[col].nunique() for col in numeric_cols],
    }
)

range_checks

,count,mean,std,min,25%,50%,75%,max
duration_months,1000.0,20.903,12.058814,4.0,12.0,18.0,24.00,72.0
credit_amount,1000.0,3271.258,2822.736876,250.0,1365.5,2319.5,3972.25,18424.0
installment_rate_pct_income,1000.0,2.973,1.118715,1.0,2.0,3.0,4.00,4.0
present_residence_years,1000.0,2.845,1.103718,1.0,2.0,3.0,4.00,4.0
age_years,1000.0,35.546,11.375469,19.0,27.0,33.0,42.00,75.0
existing_credits_count,1000.0,1.407,0.577654,1.0,1.0,1.0,2.00,4.0
liable_people_count,1000.0,1.155,0.362086,1.0,1.0,1.0,1.00,2.0


,feature,min_value,max_value,unique_values
0,duration_months,4,72,33
1,credit_amount,250,18424,921
2,installment_rate_pct_income,1,4,4
3,present_residence_years,1,4,4
4,age_years,19,75,53
5,existing_credits_count,1,4,4
6,liable_people_count,1,2,2


### Numeric Range Decision

The main numeric fields should be retained if values are plausible. Right-skewed variables such as `credit_amount` and `duration_months` should be handled later through transformation, capping, or robust modeling rather than removed during cleaning.

## 7. Review Category Labels

Inspect categorical values to check whether labels are consistent and ready for later encoding.

In [7]:
for col in categorical_cols:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False).rename("count").to_frame())


--- checking_account_status ---


,count
checking_account_status,
A14,394
A11,274
A12,269
A13,63



--- credit_history ---


,count
credit_history,
A32,530
A34,293
A33,88
A31,49
A30,40



--- purpose ---


...

,count
job_type,
A173,630
A172,200
A174,148
A171,22



--- telephone ---


,count
telephone,
A191,596
A192,404



--- foreign_worker ---


,count
foreign_worker,
A201,963
A202,37


### Category Label Decision

The categorical values should be retained as readable labels for analysis. Encoding belongs later in preprocessing so that category mappings are learned from the training split only.

## 8. Flag Outlier-Sensitive Variables

Use the IQR rule to identify variables that may need later transformation or capping. This is a review step, not a deletion step.

In [8]:
outlier_review_cols = [
    "credit_amount",
    "duration_months",
    "age_years",
    "existing_credits_count",
]

outlier_rows = []

for col in outlier_review_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    outlier_rows.append(
        {
            "feature": col,
            "q1": round(q1, 2),
            "q3": round(q3, 2),
            "iqr": round(iqr, 2),
            "lower_bound": round(lower_bound, 2),
            "upper_bound": round(upper_bound, 2),
            "outlier_count": int(outlier_mask.sum()),
            "outlier_pct": round(outlier_mask.mean() * 100, 2),
        }
    )

outlier_summary = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
outlier_summary

,feature,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,credit_amount,1365.5,3972.25,2606.75,-2544.62,7882.38,72,7.2
1,duration_months,12.0,24.00,12.00,-6.00,42.00,70,7.0
2,age_years,27.0,42.00,15.00,4.50,64.50,23,2.3
3,existing_credits_count,1.0,2.00,1.00,-0.50,3.50,6,0.6


### Outlier Cleaning Decision

`credit_amount` and `duration_months` are expected to contain genuine upper-tail lending cases. These observations should be retained during cleaning and reviewed again during preprocessing.

## 9. Define Column-Level Cleaning Rules

Create a decision table that records exactly how each important column group should be handled.

In [9]:
cleaning_plan = pd.DataFrame(
    [
        {
            "column_or_group": "applicant_id",
            "issue": "Identifier only",
            "decision": "Keep for traceability; exclude from model features",
            "stage": "split/preprocessing",
        },
        {
            "column_or_group": "risk_class, risk_label, TARGET",
            "issue": "Target representation columns",
            "decision": "Use TARGET for modeling; keep labels for interpretation only",
            "stage": "split/preprocessing",
        },
        {
            "column_or_group": "categorical features",
            "issue": "Need model-ready encoding later",
            "decision": "Retain readable labels; encode after train/test split",
            "stage": "preprocessing",
        },
        {
            "column_or_group": "credit_amount",
            "issue": "Right-skewed with upper-tail outliers",
            "decision": "Retain; consider log transform or capping later",
            "stage": "preprocessing",
        },
        {
            "column_or_group": "duration_months",
            "issue": "Right-skewed with long-duration cases",
            "decision": "Retain; review robust treatment later",
            "stage": "preprocessing",
        },
        {
            "column_or_group": "age_years",
            "issue": "Mild upper-tail outliers",
            "decision": "Retain with minimal intervention",
            "stage": "preprocessing",
        },
        {
            "column_or_group": "missing values",
            "issue": "Confirm whether any fields require imputation",
            "decision": "No cleaning action if missing counts are zero; impute only inside preprocessing if needed",
            "stage": "cleaning/preprocessing",
        },
        {
            "column_or_group": "duplicate rows",
            "issue": "Potential repeated records",
            "decision": "Remove only if exact duplicates are confirmed",
            "stage": "cleaning",
        },
    ]
)

cleaning_plan

,column_or_group,issue,decision,stage
0,applicant_id,Identifier only,Keep for traceability; exclude from model feat...,split/preprocessing
1,"risk_class, risk_label, TARGET",Target representation columns,Use TARGET for modeling; keep labels for inter...,split/preprocessing
2,categorical features,Need model-ready encoding later,Retain readable labels; encode after train/tes...,preprocessing
3,credit_amount,Right-skewed with upper-tail outliers,Retain; consider log transform or capping later,preprocessing
4,duration_months,Right-skewed with long-duration cases,Retain; review robust treatment later,preprocessing
5,age_years,Mild upper-tail outliers,Retain with minimal intervention,preprocessing
6,missing values,Confirm whether any fields require imputation,No cleaning action if missing counts are zero;...,cleaning/preprocessing
7,duplicate rows,Potential repeated records,Remove only if exact duplicates are confirmed,cleaning


## 10. Dataset-Level Cleaning Principles

These rules keep the project consistent and protect the modeling workflow from leakage.

In [10]:
dataset_rules = pd.DataFrame(
    [
        {
            "rule": "Do not drop financial outliers during cleaning",
            "reason": "Extreme lending values may be genuine risk signals",
        },
        {
            "rule": "Do not fit preprocessing transformations before splitting",
            "reason": "Fitted cleaning and preprocessing steps can leak information from validation/test data",
        },
        {
            "rule": "Keep target and identifier columns out of model features",
            "reason": "This prevents leakage and keeps model inputs realistic",
        },
        {
            "rule": "Keep categorical values readable during analysis",
            "reason": "Readable labels make stakeholder interpretation easier before encoding",
        },
    ]
)

dataset_rules

,rule,reason
0,Do not drop financial outliers during cleaning,Extreme lending values may be genuine risk sig...
1,Do not fit preprocessing transformations befor...,Fitted cleaning and preprocessing steps can le...
2,Keep target and identifier columns out of mode...,This prevents leakage and keeps model inputs r...
3,Keep categorical values readable during analysis,Readable labels make stakeholder interpretatio...


## 11. Final Cleaning Summary

The dataset is suitable to move forward into split strategy and leakage control. No major structural cleaning is required at this stage. The main decisions are to retain outlier-sensitive lending variables, keep categorical labels readable for analysis, and postpone fitted transformations until after the split.

Next notebook: `05_split_strategy_and_leakage_control.ipynb`.